# From geometry to crack propagation: a square-plate SENT simulation

This tutorial performs a **fresh phase-field fracture calculation** with PhAST
in the conventional finite-element order:

1. install and verify the software;
2. load the geometry;
3. generate and inspect the mesh and named boundaries;
4. define the material and phase-field model;
5. apply loads and boundary conditions;
6. select a supported analysis and solver route;
7. run the coupled problem with visible progress; and
8. inspect displacement, strain, stress, and damage and animate crack growth.

The model is the 40 mm by 40 mm B3 single-edge-notched tension (SENT) plate. A
V-notch extends from the left boundary to \(x=20\) mm. This is a compact
teaching calculation derived from the public B3 input, not a new benchmark
validation or a mesh/time-step convergence study.

The calculation uses a CPU; no GPU is required. Installation, meshing, solving,
and rendering have separate costs, and no portable runtime is claimed.

The B3 input retains its **schema-v1 compatibility** format to preserve the
matched public example. For the single saved schema-v2 configuration workflow,
use the [standard problem-setup walkthrough](notebook_setup.ipynb).
This notebook does not extend the schema-v2 solver capability boundary.


## 0. Install PhAST by the portable source route

Use Python >=3.10 with compatible dependency wheels. For a new source checkout:

~~~bash
git clone https://github.com/CEMS-Lab/PhAST.git
cd PhAST
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -e . jupyterlab
python run_sanitizer.py
python -m phast doctor
jupyter lab docs/tutorial/notebook_square_plate_fracture.ipynb
~~~

On Windows PowerShell, activate with <code>.venv\Scripts\Activate.ps1</code>.
Conda and Docker routes are in the [installation guide](../install.md). The base
CPU installation is sufficient; optional HPC backends are not required.

For an existing checkout or a documentation build, use that checkout and its
prepared environment. Do not clone or install an older release over it. The
clone command above selects public `main`, which is mutable; local changes
are not assumed to be published there.


In [ ]:
from __future__ import annotations
import copy
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys

from IPython.display import HTML, Image, Markdown, display
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.animation import FuncAnimation, PillowWriter
import numpy as np
import yaml
import h5py


def find_repository_root(start: Path) -> Path:
    # Locate a source checkout from the notebook working directory.
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise RuntimeError(
        "Run this notebook from a cloned PhAST repository. "
        "See the installation commands above."
    )

ROOT = find_repository_root(Path.cwd())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ["PYTHONPATH"] = str(ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")

import phast
from phast.core.mesh import FEMMesh
from phast.physics.material import Material

plt.rcParams.update({"font.family": "STIXGeneral", "mathtext.fontset": "stix"})

def require_finite(values, name):
    """Reject empty or non-finite data rather than disguising a failed solve."""
    array = np.asarray(values)
    if array.size == 0 or not np.isfinite(array).all():
        raise ValueError(
            f"Empty or non-finite values in {name}; inspect the solver log "
            "before plotting or interpreting this run."
        )
    return array

output_root = Path(os.environ.get("PHAST_NOTEBOOK_OUTPUT_ROOT", ROOT / "runs"))
base = output_root.expanduser().resolve() / "tutorial_square_plate"
RUN_DIR = base
suffix = 2
while RUN_DIR.exists():
    RUN_DIR = Path(f"{base}_{suffix}")
    suffix += 1
RUN_DIR.mkdir(parents=True)

GEO_SOURCE = Path("examples/dynamic/B3_dynamic_sent/mesh.geo")
BASE_CONFIG = Path("examples/dynamic/B3_dynamic_sent/config.yaml")
GEO_PATH = RUN_DIR / "mesh.geo"
MESH_PATH = RUN_DIR / "mesh.msh"
CONFIG_PATH = RUN_DIR / "config.yaml"

print(f"Repository: {ROOT.name}")
print(f"Tutorial output: {RUN_DIR}")
print(f"PhAST import: {Path(phast.__file__).resolve().relative_to(ROOT)}")


## 1. Load the geometry and identify its boundaries

The checked-in Gmsh file defines the square, the left-edge notch, a refinement
line along the expected crack corridor, and named physical curves. The exterior
sides are <code>top</code>, <code>bottom</code>, <code>left</code>, and
<code>right</code>; the notch faces are <code>notch_upper</code> and
<code>notch_lower</code>. These names connect geometry to boundary conditions
without coordinate-based node selection.

In [ ]:
shutil.copy2(GEO_SOURCE, GEO_PATH)
geo_text = GEO_PATH.read_text(encoding="utf-8")
print(f"Loaded geometry: {GEO_PATH}")
print("\nGeometry entities and physical groups:\n")
for line in geo_text.splitlines():
    stripped = line.strip()
    if stripped.startswith(("Point(", "Line(", "Physical Curve", "Physical Surface")):
        print(stripped)

## 2. Generate and inspect the finite-element mesh

Gmsh creates first-order triangles with refinement around the notch and expected
horizontal crack path. PhAST reads the mesh through <code>meshio</code> and
converts the physical curves into named node sets. Fracture resolution is
assessed relative to \(\ell_0\), not only by visual appearance.

In [ ]:
gmsh_executable = shutil.which("gmsh")
if gmsh_executable is None:
    raise RuntimeError(
        "Gmsh was not found. Reinstall the PhAST environment or follow "
        "the Gmsh recovery advice in docs/install.md."
    )

completed = subprocess.run(
    [gmsh_executable, str(GEO_PATH), "-2", "-format", "msh2", "-o", str(MESH_PATH)],
    cwd=ROOT,
    text=True,
    capture_output=True,
    timeout=60,
)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or completed.stdout)

mesh = FEMMesh(str(MESH_PATH), device="cpu")
nodes = require_finite(mesh.nodes.detach().cpu().numpy(), "mesh coordinates")
elements = mesh.elements.detach().cpu().numpy()
triangulation = mtri.Triangulation(nodes[:, 0], nodes[:, 1], elements)

boundary_colours = {
    "top": "#2f6f9f", "bottom": "#2f6f9f",
    "left": "#4f4f4f", "right": "#4f4f4f",
    "notch_upper": "#b24a3b", "notch_lower": "#b24a3b",
}
fig, ax = plt.subplots(figsize=(8.2, 7.0), constrained_layout=True)
ax.triplot(triangulation, color="#c8c8c8", linewidth=0.35)
for name, colour in boundary_colours.items():
    indices = mesh.node_sets[name].detach().cpu().numpy()
    points = nodes[indices]
    ax.scatter(points[:, 0], points[:, 1], s=13, color=colour, label=name, zorder=3)
ax.set(aspect="equal", xlabel="x [mm]", ylabel="y [mm]",
       title="Generated mesh and named boundary node sets")
ax.legend(ncol=3, loc="upper center", bbox_to_anchor=(0.5, -0.09), frameon=False)
plt.show()
print(mesh.summary())


## 3. Define the material and phase-field model

| Quantity | Symbol | Value |
|---|---:|---:|
| Young's modulus | \(E\) | 32,000 MPa |
| Poisson ratio | \(\nu\) | 0.20 |
| Density | \(\rho\) | \(2.45\times10^{-9}\) tonne/mm\(^3\) |
| Critical fracture energy | \(G_c\) | 0.003 N/mm |
| Phase-field length | \(\ell_0\) | 0.5 mm |
| Residual stiffness | \(\eta\) | \(10^{-7}\) |
| Model and energy split |  | AT2, spectral |
| Kinematics |  | plane strain |

Here \(d=0\) denotes intact material and \(d=1\) fully developed damage. These
parameters are retained for the teaching example; they are not calibration
recommendations for another material.

In [ ]:
source_config = yaml.safe_load(BASE_CONFIG.read_text(encoding="utf-8"))
m = source_config["material"]
material = Material(
    E=m["E"], nu=m["nu"], Gc=m["Gc"], l0=m["l0"], rho=m["rho"],
    eta_residual=m["eta_residual"], energy_split=m["energy_split"],
    pf_model=m["pf_model"], plane_stress=m.get("plane_stress", False),
)
h_min = float(mesh.h_min)
print(material)
print(f"h_min / l0 = {h_min / float(material.l0):.3f}")

## 4. Apply loads and boundary conditions

| Region | Condition |
|---|---|
| <code>left</code> | \(u_x=0\) |
| <code>right</code> | \(u_x=0\) |
| <code>top</code> | prescribed \(u_y=+0.002\) mm |
| <code>bottom</code> | prescribed \(u_y=-0.002\) mm |
| notch faces | traction-free geometric boundaries |

The vertical displacement follows a smooth ramp over the first 20 microseconds
and is then held. The signs pull the upper and lower boundaries apart.

In [ ]:
display(Markdown(
    "~~~yaml\n"
    + yaml.safe_dump(
        {"boundary_conditions": source_config["boundary_conditions"],
         "loading": source_config["loading"]},
        sort_keys=False,
    )
    + "~~~"
))

## 5. Select the analysis and solver route

Dynamic versus quasi-static describes treatment of inertia. Explicit versus
implicit describes how a subproblem is advanced or solved. They are not four
interchangeable choices.

| Route | Current public status | Use here |
|---|---|---|
| Dynamic mechanics, explicit central difference | Supported | **Selected** |
| Quasi-static mechanics and damage solves | Supported in separate examples | Not selected |
| Implicit dynamic generalized-\(\alpha\) mechanics | Not a production route | Not selectable |
| Fully explicit damage evolution | Not this classical route | Not selectable |

Mechanics is advanced explicitly with a lumped mass matrix. At every time
increment, PhAST updates the tensile history and solves the current AT2 damage
equation with matrix-free, Jacobi-preconditioned conjugate gradients. This is an
implicit damage update, not implicit dynamics.

For fixed history \(H\), the AT2 damage equation on this route is linear in
\(d\). The overall evolution remains coupled because mechanics changes \(H\)
and damage changes the degraded stiffness used next:

\[
(\mathbf{u}_n,\mathbf{v}_n,\mathbf{a}_n,d_n)
\rightarrow \mathbf{u}_{n+1}
\rightarrow H_{n+1}
\rightarrow d_{n+1}
\rightarrow (\mathbf{v}_{n+1},\mathbf{a}_{n+1}).
\]

This partitioned sequence differs from repeated alternate-minimization
iterations within a quasi-static load increment. The CFL condition controls
explicit time-step stability; the damage tolerance controls residual
convergence of the linear damage solve. One cannot replace the other.

## 6. Choose a CFL-safe step count and write the configuration

Changing only the explicit step count while retaining an unrelated time
increment is unsafe. This notebook computes

\[
\Delta t = 0.8\,h_{\min}/c_p,
\]

then selects 6,101 increments and sets the end time consistently. Final index
6,100 is divisible by the snapshot interval of 50, so the last stored field is
the actual final solver step. The portable Jacobi damage route requires no
optional sparse or HPC backend.

In [ ]:
NUM_STEPS = 6101
SNAPSHOT_EVERY = 50
PRINT_EVERY = 100
DT_SAFETY = 0.8

dt = DT_SAFETY * h_min / float(material.c_p)
t_total = NUM_STEPS * dt
assert (NUM_STEPS - 1) % SNAPSHOT_EVERY == 0

config = copy.deepcopy(source_config)
config["problem"]["name"] = "Square-plate SENT: executable teaching calculation"
config["geometry"] = {"mesh_path": "mesh.msh"}
config["loading"].update({"num_steps": NUM_STEPS, "dt": dt, "t_total": t_total})
config["solver"].update({
    "solver_type": "explicit",
    "time_integrator": "central_difference",
    "dt_safety": DT_SAFETY,
    "damage_every": 1,
    "use_multigrid": False,
    "preconditioner": "jacobi",
    "damage_max_iter": 1000,
})
config["output"].update({
    "trajectory": True,
    "h5": True,
    "trajectory_format": "h5",
    "h5_every": SNAPSHOT_EVERY,
    "print_every": PRINT_EVERY,
    "plots": False,
    "gif": False,
    "profile": True,
})
config["device"] = {"device": "cpu", "compile": False}
CONFIG_PATH.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")

print(f"Configuration: {CONFIG_PATH}")
print("Selected route: dynamic / explicit mechanics / implicit AT2 damage")
print(f"Number of steps: {NUM_STEPS}")
print(f"Stable dt: {dt:.6e} s")
print(f"End time: {t_total * 1e6:.3f} microseconds")
print(f"Stored snapshots: {(NUM_STEPS - 1) // SNAPSHOT_EVERY + 1}")

## 7. Validate, then run with actual progress

<code>--validate-only</code> checks the YAML schema; it does not solve the PDE
or establish scientific validity. The progress bar below is driven by actual
step lines emitted by PhAST, not an elapsed-time estimate. The log prints the
resolved device, integrator, phase-field model, history update, damage route,
bounds treatment, and preconditioner.

In [ ]:
preflight = subprocess.run(
    [sys.executable, "-m", "phast", "run", str(CONFIG_PATH), "--validate-only"],
    cwd=ROOT, text=True, capture_output=True, timeout=60,
)
print(preflight.stdout, end="")
print(preflight.stderr, end="")
if preflight.returncode != 0:
    raise RuntimeError("Configuration preflight failed.")


In [ ]:
if any((RUN_DIR / name).exists() for name in ("training_data.h5", "training_data.zarr")):
    raise RuntimeError(
        f"{RUN_DIR} already contains a trajectory. Restart the kernel to create "
        "a new numbered directory rather than overwriting evidence."
    )

run_command = [
    sys.executable, "-m", "phast", "run", str(CONFIG_PATH),
    "--device", "cpu", "--output_dir", str(RUN_DIR),
]
progress = display(
    HTML(f"<progress value='0' max='{NUM_STEPS}' style='width:70%'></progress> "
         f"0 / {NUM_STEPS} steps"),
    display_id=True,
)
process = subprocess.Popen(
    run_command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
    match = re.search(r"Step\s+(\d+)", line)
    if match:
        completed_step = int(match.group(1)) + 1
        progress.update(
            HTML(f"<progress value='{completed_step}' max='{NUM_STEPS}' "
                 f"style='width:70%'></progress> "
                 f"{completed_step:,} / {NUM_STEPS:,} steps")
        )
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"PhAST exited with status {return_code}.")
progress.update(
    HTML(f"<progress value='{NUM_STEPS}' max='{NUM_STEPS}' "
         f"style='width:70%'></progress> "
         f"{NUM_STEPS:,} / {NUM_STEPS:,} steps complete")
)

## 8. Load and check the stored fields

Displacement and damage are nodal fields. Strain and stress, when stored, are
element fields. The checks require final stored step 6,100 and reject empty or
non-finite arrays before plotting. Missing optional strain or stress output
is reported, not replaced by a fabricated field.


In [ ]:
result = phast.load_result(RUN_DIR)
print("Stored fields:", result.field_names())

final_step = NUM_STEPS - 1
final_fields = {}
for name in ("displacement", "strain", "stress", "damage"):
    if not result.has_field(name):
        if name in {"displacement", "damage"}:
            raise RuntimeError(f"Required {name} trajectory is missing.")
        print(f"Optional {name} output was not stored; its plot will be omitted.")
        continue
    field = require_finite(result.field(name, step=final_step), name)
    final_fields[name] = field
    print(f"{name:12s} shape={field.shape}, min={field.min():.6g}, max={field.max():.6g}")

displacement = final_fields["displacement"]
damage = final_fields["damage"]
strain = final_fields.get("strain")
stress = final_fields.get("stress")

with h5py.File(RUN_DIR / "training_data.h5", "r") as store:
    simulation = store["simulation_data"]
    snapshots = simulation["steps"]
    step_names = sorted(snapshots, key=lambda name: int(name.rsplit("_", 1)[-1]))
    stored_steps = np.asarray([int(name.rsplit("_", 1)[-1]) for name in step_names])
    stored_times = require_finite(
        np.asarray([snapshots[name].attrs["time_s"] for name in step_names]),
        "snapshot times",
    )
    coordinates = require_finite(
        np.asarray(simulation["mesh"]["node_coordinates"]), "stored mesh coordinates"
    )
    connectivity = np.asarray(simulation["mesh"]["element_connectivity"])
if not len(stored_steps) or int(stored_steps[-1]) != final_step:
    raise RuntimeError("The final solver step is not present in the trajectory.")
if displacement.shape != (len(coordinates), 2) or damage.shape != (len(coordinates),):
    raise ValueError("Stored nodal fields do not match the mesh.")
for name, field in (("strain", strain), ("stress", stress)):
    if field is not None and field.shape != (len(connectivity), 3):
        raise ValueError(f"Unexpected in-plane {name} shape: {field.shape}")

plot_mesh = mtri.Triangulation(coordinates[:, 0], coordinates[:, 1], connectivity)


## 9. Visualise the available final fields

The views show displacement magnitude, vertical strain, an **in-plane
equivalent stress**, and phase-field damage. The stress scalar uses only the
stored \((\sigma_{xx},\sigma_{yy},\sigma_{xy})\) components:

\[
\sigma_{\mathrm{eq,in\mbox{-}plane}}
= \sqrt{\sigma_{xx}^{2}-\sigma_{xx}\sigma_{yy}
        +\sigma_{yy}^{2}+3\sigma_{xy}^{2}}.
\]

This is **not the full three-dimensional von Mises stress for plane strain**.
The out-of-plane stress \(\sigma_{zz}\) is not included in these stored fields;
it is not reconstructed here, particularly with the selected spectral
tension/compression split and damage degradation. This scalar is a labelled
visualisation, not a yield or failure criterion.

Element fields are rendered elementwise. Both the static damage plot and the
animation use `Reds` with the fixed scale \(0\leq d\leq1\).


In [ ]:
u_magnitude = require_finite(np.linalg.norm(displacement, axis=1), "displacement magnitude")
epsilon_yy = None if strain is None else strain[:, 1]
in_plane_equivalent_stress = None
if stress is not None:
    sigma_xx, sigma_yy, sigma_xy = stress.T
    in_plane_equivalent_stress = require_finite(
        np.sqrt(np.maximum(
            sigma_xx**2 - sigma_xx * sigma_yy + sigma_yy**2 + 3.0 * sigma_xy**2,
            0.0,
        )),
        "in-plane equivalent stress",
    )

fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
specs = [
    ("Displacement magnitude, |u| [mm]", u_magnitude, "cividis", "nodal"),
    ("Vertical strain, epsilon_yy [-]", epsilon_yy, "RdBu_r", "element"),
    ("In-plane equivalent stress [MPa]", in_plane_equivalent_stress, "viridis", "element"),
    ("Phase-field damage, d [-]", damage, "Reds", "nodal"),
]
for ax, (title, field, cmap, location) in zip(axes.flat, specs):
    if field is None:
        ax.text(0.5, 0.5, f"{title}\nOptional field not stored",
                ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
        continue
    field = require_finite(field, title)
    limits = dict(vmin=0.0, vmax=1.0) if cmap == "Reds" else {}
    artist = (
        ax.tripcolor(plot_mesh, field, shading="gouraud", cmap=cmap, **limits)
        if location == "nodal"
        else ax.tripcolor(plot_mesh, facecolors=field, shading="flat", cmap=cmap, **limits)
    )
    ax.set(aspect="equal", xlabel="x [mm]", ylabel="y [mm]", title=title)
    fig.colorbar(artist, ax=ax, shrink=0.82)
fig.suptitle(
    f"Square-plate SENT: step {final_step} at {stored_times[-1] * 1e6:.2f} microseconds",
    fontsize=14,
)
field_figure = RUN_DIR / "final_fields.png"
fig.savefig(field_figure, dpi=160)
plt.show()
print(f"Saved: {field_figure}")


## 10. Animate the stored damage evolution

The geometric notch terminates at \(x=20\) mm. Inspect whether and where the
stored high-damage band advances. Every selected frame is checked for finite
values before rendering. An animation is not evidence of mesh convergence,
time-step convergence, or experimental agreement.


In [ ]:
frame_indices = np.unique(np.linspace(0, len(stored_steps) - 1, 62, dtype=int))
damage_frames = [
    require_finite(result.field("damage", step=int(stored_steps[index])),
                   f"damage at step {stored_steps[index]}")
    for index in frame_indices
]
if any(field.shape != (len(coordinates),) for field in damage_frames):
    raise ValueError("A damage frame does not match the mesh nodes.")
fig, ax = plt.subplots(figsize=(7.2, 6.2), constrained_layout=True)
artist = ax.tripcolor(
    plot_mesh, damage_frames[0],
    shading="gouraud", cmap="Reds", vmin=0.0, vmax=1.0,
)
ax.set(aspect="equal", xlabel="x [mm]", ylabel="y [mm]")
fig.colorbar(artist, ax=ax).set_label("damage d")

def update_animation(frame_number: int):
    index = frame_indices[frame_number]
    artist.set_array(damage_frames[frame_number])
    ax.set_title(
        f"Crack evolution: step {stored_steps[index]} | "
        f"t = {stored_times[index] * 1e6:.2f} microseconds"
    )
    return (artist,)

animation = FuncAnimation(
    fig, update_animation, frames=len(frame_indices), interval=100, blit=False
)
animation_path = RUN_DIR / "crack_evolution.gif"
animation.save(animation_path, writer=PillowWriter(fps=10), dpi=115)
plt.close(fig)
display(Image(filename=str(animation_path)))
print(f"Saved: {animation_path}")


## 11. Summarize and retain the result

The \(x\)-extent of nodes with \(d\geq0.95\) is descriptive. It does not replace
a crack-tip algorithm, energy balance, convergence study, or experiment.

In [ ]:
high_damage_nodes = coordinates[damage >= 0.95]
if len(high_damage_nodes):
    print(
        "Nodes with d >= 0.95 span "
        f"x = {high_damage_nodes[:, 0].min():.3f} to "
        f"{high_damage_nodes[:, 0].max():.3f} mm."
    )
print("\nTop-level run artifacts:")
for artifact in sorted(RUN_DIR.iterdir()):
    if artifact.name != "training_data.h5":
        print(f"  {artifact.name}")
print("  training_data.h5  (single-file reloadable trajectory)")

## Interpretation and limitations

Successful execution demonstrates this workflow for the selected configuration.
The cells report which fields were stored, check their finiteness, and print
the actual extent of nodes exceeding the stated damage threshold. Cleared
notebook outputs make no claim about a previously computed crack path.

Do not infer more than that:

- the mesh and time increment have not undergone a convergence study;
- the result is not newly compared with the Borden et al. benchmark data;
- the compact mesh is instructional, not a publication-quality prediction;
- local CPU timing is not a portable performance claim;
- changing \(G_c\), \(\ell_0\), loading, or mesh defines a new study and
  requires corresponding resolution and validation checks.

For quasi-static fracture, use
<code>examples/quasistatic/miehe_tension/config.yaml</code>. Do not change only
the solver type here and assume the boundary-value problem remains equivalent.

## Suggested student exercise

Predict first, modify one item, and rerun into a new directory:

- increase <code>SNAPSHOT_EVERY</code> to reduce stored trajectory size without
  changing the solver update cadence; or
- refine the mesh and recompute the CFL increment before changing the step count.

If installation, execution, or interpretation remains unclear, open a
[GitHub issue](https://github.com/CEMS-Lab/PhAST/issues) with the first error,
platform, Python version, command, and configuration. Questions, reproducible
bug reports, documentation corrections, and student contributions are welcome.
